# Solvey: QLoRA SFT of Qwen2.5-VL-3B-Instruct on IT-support screenshots + text

**Environment:** Kaggle, GPU = **T4 x1 or x2** (the code uses GPU 0 only), Internet = ON.

**Run order**
1. Run the **install cell once** on a fresh session, then **Restart session** (Run > Restart).
2. After the restart, **skip the install cell** and run everything from "Check versions" down.

Design choices that avoid the usual breakages: pinned versions, no `trl`, no `datasets` (plain Python lists),
fp16 (T4 has no bf16), no FlashAttention, capped image resolution.

## 1. Install (run once, then restart the session)

In [ ]:
!pip install -q "transformers==4.57.1" "peft==0.17.1" "accelerate==1.10.1" "bitsandbytes>=0.46.1" "datasets==3.6.0" "pyarrow==19.0.1" beautifulsoup4 pillow

**Now restart the session (Run > Restart session) and continue from the next cell. Do not run the install cell again.**

## 2. Check versions

In [ ]:
import os
DEBUG_SYNC = False   # set True ONLY if a CUDA error persists: exact failing line, but slow
if DEBUG_SYNC:
    os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch, transformers, peft, accelerate, bitsandbytes, pyarrow
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| accelerate", accelerate.__version__, "| bnb", bitsandbytes.__version__, "| pyarrow", pyarrow.__version__)
assert torch.cuda.is_available(), "Turn on the GPU: Settings > Accelerator > GPU T4"
print("GPU:", torch.cuda.get_device_name(0), "| GPUs visible:", torch.cuda.device_count())

## 3. Config (edit the values in CAPS)

In [ ]:
import os, re, json, ast, glob, random, gc
from PIL import Image

BASE = "/kaggle/input/datasets/mariambenabdallah/stack-exchange-api-results"
CSV_PATH = f"{BASE}/so_dataset.csv"
IMG_DIR = f"{BASE}/images/images"

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
OUT_DIR = "/kaggle/working/solvey-lora"
HF_REPO = "HackedM/solvey-qwen2.5-vl-3b-lora"     # <-- change

MAX_IMG_TOKENS = 512        # visual tokens per image. Lower to 384 / 256 if you hit OOM.
MAX_ANSWER_CHARS = 1500     # answer cap (fewer tokens, less rambling)
MAX_QUESTION_CHARS = 1500
EPOCHS = 2
LR = 1e-4
FILTER_NEEDS_REVIEW = False  # True = drop rows where needs_review is True
ATTN_IMPL = "eager"         # safest on T4. Once training works you can try "sdpa" for speed.
OPTIM = "adamw_torch"       # plain AdamW: LoRA params are tiny; paged 8-bit optimizers are a known cause of CUDA illegal-memory-access

# The SAME system prompt is used in training and inference. Don't change it after training.
SYSTEM_PROMPT = """You are Solvey, an IT support assistant. The user gives you a description of a problem and usually a screenshot (error message, terminal, settings window, etc.).
Respond in this exact format:
Category: <category> / <subcategory>
Difficulty: <easy|medium|hard>
Solution:
<clear numbered steps that fix the problem>
Be concrete. Quote the exact error text you see in the screenshot. If the screenshot is missing or unreadable, say so and rely on the text."""

## 4. Load processor + 4-bit model

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=64 * 28 * 28,
    max_pixels=MAX_IMG_TOKENS * 28 * 28,
)
processor.tokenizer.padding_side = "right"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map={"": 0},          # single GPU (avoids slow naive model-parallel on T4 x2)
    torch_dtype=torch.float16,
    attn_implementation=ATTN_IMPL,
)
print("Loaded:", type(model).__name__, "| GPU mem:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")

## 5. Build the dataset from the CSV + images

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup

df = pd.read_csv(CSV_PATH)
print("CSV shape:", df.shape)
print("Raw image_files examples:", df.loc[df["has_images"].astype(str).str.lower().isin(["true", "1"]), "image_files"].head(3).tolist())

def html2text(h):
    if not isinstance(h, str):
        return ""
    s = BeautifulSoup(h, "html.parser")
    for pre in s.find_all("pre"):
        pre.replace_with("\n```\n" + pre.get_text() + "\n```\n")
    return re.sub(r"\n{3,}", "\n\n", s.get_text()).strip()

def txt(x, default):
    return x.strip() if isinstance(x, str) and x.strip() else default

def parse_files(v):
    if not isinstance(v, str) or not v.strip():
        return []
    try:
        x = json.loads(v)
    except Exception:
        try:
            x = ast.literal_eval(v)
        except Exception:
            x = re.split(r"[;,|]\s*", v)
    x = x if isinstance(x, list) else [x]
    return [os.path.basename(str(f)).strip() for f in x if str(f).strip()]

def usable_image(p):
    try:
        with Image.open(p) as im:
            w, h = im.size
        return w >= 28 and h >= 28 and max(w / h, h / w) < 50
    except Exception:
        return False

def find_image(r):
    paths = [f"{IMG_DIR}/{n}" for n in parse_files(r.image_files)]
    paths = [p for p in paths if os.path.exists(p)]
    if not paths:  # fallback: match by question id
        try:
            paths = sorted(glob.glob(f"{IMG_DIR}/so_{int(r.question_id)}_*"))
        except Exception:
            paths = []
    paths = [p for p in paths if usable_image(p)]
    return paths[0] if paths else None   # 1 image per sample (memory)

rows = []
for r in df.itertuples():
    if str(r.has_answer).lower() not in ("true", "1", "1.0"):
        continue
    if FILTER_NEEDS_REVIEW and str(r.needs_review).lower() in ("true", "1", "1.0"):
        continue
    answer = r.resolution if isinstance(r.resolution, str) and len(r.resolution) > 30 else html2text(r.answer_body_html)
    answer = answer[:MAX_ANSWER_CHARS].strip()
    if len(answer) < 30:
        continue
    user_text = (txt(r.title, "") + "\n\n" + html2text(r.body_html)[:MAX_QUESTION_CHARS]).strip()
    target = (f"Category: {txt(r.category, 'other')} / {txt(r.subcategory, 'other')}\n"
              f"Difficulty: {txt(r.difficulty, 'medium')}\n"
              f"Solution:\n{answer}")
    rows.append({"image": find_image(r), "user_text": user_text, "target": target})

random.seed(42)
random.shuffle(rows)
n_test = max(20, int(len(rows) * 0.1))
ds = {"train": rows[n_test:], "test": rows[:n_test]}
print(f"train={len(ds['train'])} test={len(ds['test'])} | with image: {sum(x['image'] is not None for x in rows)}/{len(rows)}")
assert sum(x['image'] is not None for x in rows) > 0, "No images matched: check IMG_DIR / the raw image_files examples printed above."

## 6. LoRA (language model only, vision tower frozen)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
model.config.use_cache = False

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    # regex: only the LLM's linear layers (the vision tower also has gate_proj/up_proj/down_proj, so a plain list would hit it)
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)|model\.layers.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print("trainable tensors:", len(trainable), "| any in vision tower:", any("visual" in n for n in trainable))

## 7. Chat formatting + collator (loss on the answer only)

In [ ]:
def build_messages(ex, with_answer):
    content = []
    if ex["image"]:
        content.append({"type": "image"})
    content.append({"type": "text", "text": ex["user_text"]})
    msgs = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": content},
    ]
    if with_answer:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": ex["target"]}]})
    return msgs

def collate(batch):
    fulls, prompts, images = [], [], []
    for ex in batch:
        fulls.append(processor.apply_chat_template(build_messages(ex, True), tokenize=False))
        prompts.append(processor.apply_chat_template(build_messages(ex, False), tokenize=False, add_generation_prompt=True))
        if ex["image"]:
            images.append(Image.open(ex["image"]).convert("RGB"))
    kw = {"images": images} if images else {}
    # no truncation on purpose: truncating would cut image tokens and crash the model
    enc = processor(text=fulls, padding=True, return_tensors="pt", **kw)
    pr = processor(text=prompts, padding=True, return_tensors="pt", **kw)
    labels = enc["input_ids"].clone()
    for i in range(len(batch)):
        labels[i, : int(pr["attention_mask"][i].sum())] = -100   # mask system + user + image
    labels[enc["attention_mask"] == 0] = -100
    enc["labels"] = labels
    return enc

# sanity check BEFORE the long run
b = collate([ds["train"][0]])
print({k: tuple(v.shape) for k, v in b.items()})
print("seq len:", b["input_ids"].shape[1], "| supervised tokens:", int((b["labels"] != -100).sum()))
print("--- supervised text ---")
print(processor.tokenizer.decode(b["labels"][0][b["labels"][0] != -100])[:600])

## 7b. Smoke test: one forward + backward per sample type (run before training)
If this passes, the model/data path is fine. If it crashes, the error is in the model forward/backward, not the optimizer.

In [ ]:
model.train()
dev = model.device
img_s = next((x for x in ds["train"] if x["image"]), None)
txt_s = next((x for x in ds["train"] if not x["image"]), None)
for name, smp in [("image+text", img_s), ("text only", txt_s)]:
    if smp is None:
        print(name, "-> no such sample, skipped")
        continue
    batch = {k: v.to(dev) for k, v in collate([smp]).items()}
    with torch.autocast("cuda", dtype=torch.float16):
        out = model(**batch)
    out.loss.backward()
    torch.cuda.synchronize()
    print(name, "OK | loss", round(out.loss.item(), 3), "| seq len", batch["input_ids"].shape[1],
          "| peak GPU GB", round(torch.cuda.max_memory_allocated() / 1024**3, 2))
    model.zero_grad(set_to_none=True)
    del batch, out
    torch.cuda.empty_cache()

## 8. Train (about 1.5-3 h on a T4; checkpoints are saved every 40 steps)

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir=OUT_DIR + "-ckpt",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=0.3,
    fp16=True,                      # T4: fp16, NOT bf16
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim=OPTIM,
    logging_steps=5,
    eval_strategy="steps", eval_steps=40,
    save_strategy="steps", save_steps=40, save_total_limit=2,
    remove_unused_columns=False,    # collator reads the raw dict columns
    label_names=["labels"],
    dataloader_num_workers=0,
    report_to="none",
)
trainer = Trainer(
    model=model, args=args,
    train_dataset=ds["train"], eval_dataset=ds["test"][:30],
    data_collator=collate,
)
trainer.train()
trainer.save_model(OUT_DIR)
processor.save_pretrained(OUT_DIR)
print("Adapter saved to", OUT_DIR)

## 9. Test the fine-tuned model

In [ ]:
def ask(text, image_path=None, max_new_tokens=450):
    ex = {"image": image_path, "user_text": text, "target": ""}
    prompt = processor.apply_chat_template(build_messages(ex, False), tokenize=False, add_generation_prompt=True)
    kw = {"images": [Image.open(image_path).convert("RGB")]} if image_path else {}
    inp = processor(text=[prompt], return_tensors="pt", **kw).to(model.device)
    model.eval()
    model.config.use_cache = True
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=0.2, top_p=0.9, repetition_penalty=1.05)
    return processor.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0]

for ex in ds["test"][:3]:
    print("QUESTION:", ex["user_text"][:300].replace("\n", " "), "| image:", ex["image"] is not None)
    print("MODEL:\n", ask(ex["user_text"], ex["image"]))
    print("REFERENCE:\n", ex["target"][:400])
    print("=" * 80)

## 10. Push to Hugging Face
Add your token as a Kaggle secret named `HF_TOKEN` (Add-ons > Secrets). Edit `HF_REPO` in the config cell first.

In [ ]:
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    from getpass import getpass
    hf_token = getpass("Paste your HF token: ")
login(hf_token)

model.push_to_hub(HF_REPO, private=True)        # LoRA adapter only (~100 MB)
processor.push_to_hub(HF_REPO, private=True)
print("Pushed adapter to https://huggingface.co/" + HF_REPO)

## 11. (Optional) Merge adapter into a standalone fp16 model and push it
Needed for Inference Endpoints. Uses CPU RAM only. Set `MERGE = True` to run.

In [ ]:
MERGE = False
MERGED_REPO = HF_REPO.replace("-lora", "-merged")

if MERGE:
    from peft import PeftModel
    base = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="cpu")
    merged = PeftModel.from_pretrained(base, OUT_DIR).merge_and_unload()
    merged.push_to_hub(MERGED_REPO, private=True)
    processor.push_to_hub(MERGED_REPO, private=True)
    print("Pushed merged model to https://huggingface.co/" + MERGED_REPO)

## 12. Using it in your Solvey backend (new session / your own server)

```python
import torch
from PIL import Image
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from peft import PeftModel

BASE_ID, ADAPTER = "Qwen/Qwen2.5-VL-3B-Instruct", "HackedM/solvey-qwen2.5-vl-3b-lora"
processor = AutoProcessor.from_pretrained(ADAPTER, min_pixels=64*28*28, max_pixels=512*28*28)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(BASE_ID, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, ADAPTER).eval()
# then reuse SYSTEM_PROMPT, build_messages() and ask() from section 9, with the identical system prompt.
```

Serving options: Hugging Face **Inference Endpoints** (use the merged repo), a **Space** with a GPU, or your own GPU server.
The free serverless Inference API generally does not host private fine-tuned vision models.